<a id="inicio"></a>

# Cinema: Serverless AWS Application

## Objective

Build and deploy an event-driven serverless application on AWS that periodically polls a third-party movies API, persists raw responses to S3, maintains a queryable DynamoDB catalog, and exposes search endpoints through API Gateway.

## Methodology

- **Data acquisition**: Call [TMDb's `now_playing` endpoint](https://www.themoviedb.org/documentation/api) via Python `requests` and Lambda
- **Object storage**: Persist each API response as a structured JSON object in S3, partitioned by movie ID and date
- **Event-driven ingest**: S3 `ObjectCreated` events trigger a Lambda that upserts key fields into DynamoDB
- **NoSQL catalog**: DynamoDB table `MoviesDB` with a Global Secondary Index on `(year_month, rating)` for range queries
- **Scheduled polling**: EventBridge rule triggers the data-acquisition Lambda on a 24-hour cadence
- **Query API**: API Gateway REST API exposing two routes — `GET /list/{year}/{month}` and `GET /movies/{id}` — backed by Lambdas that query DynamoDB
- **Ops & validation**: IPyWidgets UI that hits the deployed API, plus a CloudWatch dashboard for monitoring

## Architecture

![Architecture](img/arquitectura.png)

<div class="alert alert-block alert-info">

### ℹ️ Note on cell outputs

The cell outputs visible in this notebook were captured during the **original run** and are shown here as a visual reference for portfolio review. Some artifacts may still appear in Spanish because the outputs predate the English code translation.

**To re-run locally:**

1. You'll need an AWS account with permissions for S3, DynamoDB, Lambda, API Gateway, CloudWatch, and EventBridge. In a coursework setting this notebook ran on AWS Academy.
2. Set up credentials using any of the standard AWS credential mechanisms — **do not paste temporary session tokens into the notebook**:
   - `aws configure` to populate `~/.aws/credentials`
   - `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN` env vars
   - IAM role if running on EC2 / SageMaker
3. Obtain a [TMDb API Key](https://www.themoviedb.org/settings/api) and set it as an environment variable: `export TMDB_API_KEY=your_key_here`
4. Replace the S3 bucket name in the cells below with your own bucket (bucket names must be globally unique).
5. Deploy each Lambda manually through the AWS Console or use Infrastructure-as-Code (SAM / CDK / Terraform) — the notebook code cells show the logic to package into each function.
6. After deploying API Gateway, update the `BASE_URL` variable near the end of the notebook with your own endpoint.

**⚠️ AWS Academy concurrency note:** The original coursework ran under AWS Academy's 10-concurrent-Lambda limit. When deploying, set each function's reserved concurrency to 1 to avoid overrun.

</div>

---

---

<a id="indice"></a>
## Contents

* [1. Introduction](#section1)
* [2. Architecture Overview](#section2)
* [3. Implementation Steps](#section3)

---

In [76]:
import os
session = boto3.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=os.environ.get("AWS_SESSION_TOKEN")  # optional for temporary credentials,
    region_name='us-east-1'
)

dynamodb = session.resource('dynamodb')

sts_client = session.client('sts')
identity = sts_client.get_caller_identity()
print(identity)

{'UserId': 'AROAZ5GCAZNQ4TLIGE533:user4107862=LuisFernando.Nunez', 'Account': '681159215969', 'Arn': 'arn:aws:sts::681159215969:assumed-role/voclabs/user4107862=LuisFernando.Nunez', 'ResponseMetadata': {'RequestId': '5bb6873b-d014-499b-a25f-3582c531915d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '5bb6873b-d014-499b-a25f-3582c531915d', 'content-type': 'text/xml', 'content-length': '474', 'date': 'Wed, 28 May 2025 08:44:51 GMT'}, 'RetryAttempts': 0}}


<a id="section1"></a>
# 1. Introduction

This project exercises several AWS services together in a small, realistic serverless application.

The scope is deliberately practical: deploy a cloud-native app that periodically queries current cinema listings via a third-party API, persists the data in AWS storage, and exposes a query API for consuming it. The core query layer is backed by DynamoDB and accessed through Lambda-backed endpoints on API Gateway.

Validation can be done interactively from this notebook using IPyWidgets.

<div style="text-align: right">
<a href="#inicio">[back to top]</a>
</div>

---

<a id="section2"></a>
# 2. Architecture Overview

The deployed application performs the following steps:

<ol type="A">
    <li><b>External API queries.</b> Fetches now-playing movie data from a third-party API. The initial prototyping happens via the <code>requests</code> library in this notebook.</li>
    <li><b>Raw persistence in S3.</b> Each API response is stored as a <code>.json</code> object, keyed with a date-based path for easy chronological lookup.</li>
    <li><b>Structured catalog in DynamoDB.</b> Movie metadata is synced into a DynamoDB table. The sync is driven by an event handler that fires on new S3 objects.</li>
    <li>
        <b>Query the catalog</b> for movies matching common filters:
        <ol>
            <li>Movies released in a specific month</li>
            <li>Movies with a rating above a threshold</li>
        </ol>
    </li>
    <li><b>Scheduled ingestion.</b> The API-to-S3 collection runs on a schedule via an EventBridge-triggered Lambda.</li>
    <li><b>Query API.</b> Exposes read endpoints through API Gateway + Lambda.</li>
</ol>

Architecture diagram:

<img src="img/arquitectura.png" alt="Application architecture">

<div style="text-align: right">
<a href="#inicio">[back to top]</a>
</div>

---

<a id="section3"></a>
# 3. Implementation Steps

Each step below extends the app's functionality incrementally.

### Step 0: TMDb API access

This app uses the [TMDb](https://www.themoviedb.org) *now_playing* endpoint. You'll need an API key:

1. Create a TMDb account and verify it
2. Navigate to Settings → API and fill out the application form
3. Copy the v3 API key for use below (or, preferably, set it as the `TMDB_API_KEY` environment variable)

The *now_playing* endpoint returns, per movie:

- `poster_path` — relative path for poster URL construction
- `adult` — boolean flag for adult content
- `overview` — plot summary
- `release_date` — release date string
- `id` — TMDb unique identifier
- `popularity` — rolling popularity score
- `vote_average` — average user vote
- `vote_count` — total number of votes cast
- ...plus additional fields documented at `https://api.themoviedb.org/3/movie/now_playing`

### Validation — API call

Using a tool like Postman (or `curl`), hit:

```
https://api.themoviedb.org/3/movie/now_playing?api_key=<YOUR_API_KEY>
```

...and review the first 3 movies in the response.

### Step 1: Python-based API query

Call the same endpoint from Python with `requests`. Add a `rank` field to each result based on its position in the response (which is already sorted by popularity):

- Position 0 → rank 1
- Position 1 → rank 2
- ...
- Position n-1 → rank n

In [64]:
import os
import requests

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
response = requests.get(f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1")

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1

    # Mostrar las 3 primeras películas con el ranking añadido
    for pelicula in peliculas[:3]:
        print(f"{pelicula['rank']}. {pelicula['title']} - Popularidad: {pelicula['popularity']}")
else:
    print(f"Error en la consulta: {response.status_code}")

1. Una película de Minecraft - Popularidad: 634.0126
2. Lilo y Stitch - Popularidad: 719.8697
3. Destino final: Lazos de sangre - Popularidad: 488.19


### Validation

Print the first 3 movies (including the `rank` field you added).

### Step 2: Upload responses to S3

- Create an S3 bucket named `mcidaen5.capstone11.movies.raw.data.<your-name>` from the AWS Console
- Modify the code below so that for each movie in the API response, a separate JSON object is uploaded to S3
- Object path format: `/movies/{ID_MOVIE}/{YEAR_MONTH_DAY}.json`
  - `ID_MOVIE` — the movie's TMDb ID
  - `YEAR/MONTH/DAY` — the query date

For example, if the API returns 4 movies, executing this cell should upload 4 JSON files (via `boto3`) into the bucket.

In [65]:
import os
import requests
import json
import boto3    
from datetime import datetime

NOMBRE_BUCKET = os.environ.get("S3_BUCKET_NAME", "<YOUR_BUCKET_NAME>")# Creado desde la consola de AWS 
now = datetime.now()
year_month_day = now.strftime("%Y_%m_%d") # Obtener dígitos del año, mes y día


# 1.- OBTENER DATOS DEL API MOVIES DB (COPIAD DEL EJERCICIO 1)

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
response = requests.get(f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1")

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1

    # Mostrar las 3 primeras películas con el ranking añadido
    for pelicula in peliculas[:3]:
        print(f"{pelicula['rank']}. {pelicula['title']} - Popularidad: {pelicula['popularity']}")
else:
    print(f"Error en la consulta: {response.status_code}")
    
# 2.- CONFIGURAD EL CLIENTE Y RECURSO DE ACCESO A S3, Y VALIDAD SU USO (CREACIÓN DE CARPETA)
    
s3_client = session.client('s3', region_name='us-east-1')
s3_resource = session.resource('s3', region_name='us-east-1')

try:
    s3_client.put_object(Bucket=NOMBRE_BUCKET, Key='movies/')
    print('Carpeta creada correctamente')
except Exception as e:
    print(e) 


# 3.- SUBID A S3 UN OBJETO POR RESULTADO OBTENIDO EN EL PASO 1

for pelicula in peliculas:
    movie_id = pelicula['id']
    ruta_s3 = f"movies/{movie_id}/{year_month_day}.json"
    contenido_json = json.dumps(pelicula, ensure_ascii=False)

    try:
        s3_client.put_object(
            Bucket=NOMBRE_BUCKET,
            Key=ruta_s3,
            Body=contenido_json.encode('utf-8'),
            ContentType='application/json'
        )
        print(f"{ruta_s3} subido correctamente.")
    except Exception as e:
        print(f"Error al subir {ruta_s3}: {e}")

1. Una película de Minecraft - Popularidad: 634.0126
2. Lilo y Stitch - Popularidad: 719.8697
3. Destino final: Lazos de sangre - Popularidad: 488.19
Carpeta creada correctamente
movies/950387/2025_05_27.json subido correctamente.
movies/552524/2025_05_27.json subido correctamente.
movies/574475/2025_05_27.json subido correctamente.
movies/1232546/2025_05_27.json subido correctamente.
movies/575265/2025_05_27.json subido correctamente.
movies/896536/2025_05_27.json subido correctamente.
movies/447273/2025_05_27.json subido correctamente.
movies/1241436/2025_05_27.json subido correctamente.
movies/1098006/2025_05_27.json subido correctamente.
movies/1001414/2025_05_27.json subido correctamente.
movies/977294/2025_05_27.json subido correctamente.
movies/1297028/2025_05_27.json subido correctamente.
movies/986056/2025_05_27.json subido correctamente.
movies/1233069/2025_05_27.json subido correctamente.
movies/1094473/2025_05_27.json subido correctamente.
movies/1144430/2025_05_27.json sub

### Validation

Run the code above and confirm new S3 objects appear. Capture a screenshot of the `movies` folder contents in S3 and save it to `img/eje_2.png`.

<img src="img/eje_2.png" alt="Validacion 2">

### Step 3: DynamoDB schema

For efficient querying, store key movie fields in DynamoDB. Create `MoviesDB` via the AWS Console with:

- **Table name**: `MoviesDB`
- **Partition Key**: `id` (string)
- **Global Secondary Index**:
  - Partition Key: `y_m` (string) — composed of 4-digit year + underscore + 2-digit month, e.g. `2025_01`
  - Sort Key: `val` (number) — the movie's average rating (`vote_average`)

Populate the table with one test entry using `boto3`. The cell below implements that.

In [66]:
import os
# CELDA 3.1

import requests
import json
import boto3    
from decimal import Decimal
from datetime import datetime

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
NOMBRE_TABLA = "MoviesDB"

# 1.- OBTENER DATOS DEL API MOVIES DB
response = requests.get(f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1")

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1

    # Mostrar las 3 primeras películas con el ranking añadido
    for pelicula in peliculas[:3]:
        print(f"{pelicula['rank']}. {pelicula['title']} - Popularidad: {pelicula['popularity']}")
else:
    print(f"Error en la consulta: {response.status_code}")
    peliculas = []

# 2.- CONFIGURAD EL RECURSO Y TABLA DE DYNAMO_DB
dynamodb_resource = session.resource('dynamodb', region_name='us-east-1')
dynamodb_table = dynamodb_resource.Table(NOMBRE_TABLA)

# 3.- MODIFICAD FORMATO PARA DINAMODB
if peliculas:
    first_movie = peliculas[0]
    
    first_movie["id"] = str(first_movie["id"])
    
    release_date = first_movie.get("release_date", "0000-00-00")
    try:
        fecha = datetime.strptime(release_date, "%Y-%m-%d")
        y_m = fecha.strftime("%Y_%m")
    except ValueError:
        y_m = "unknown"
        
    first_movie["y_m"] = y_m
    first_movie["val"] = first_movie.get("vote_average", 0)

    # 4.- GUARDAR EN DYNAMO DB
    dynamodb_table.put_item(Item=json.loads(json.dumps(first_movie), parse_float=Decimal))
    print("✅ Película insertada correctamente en DynamoDB.")

1. Una película de Minecraft - Popularidad: 634.0126
2. Lilo y Stitch - Popularidad: 719.8697
3. Destino final: Lazos de sangre - Popularidad: 488.19
✅ Película insertada correctamente en DynamoDB.


### Validation

Run the code and confirm an entry appears in DynamoDB. Capture a screenshot of the `MoviesDB` table and save it to `img/eje_3.png`.

<img src="img/eje_3.png" alt="Validacion 3">

<div class="alert alert-block alert-info">
⚠️ You may need to coerce some field types so they can be used as partition or sort keys (DynamoDB is strict about key types).
</div>

### Step 4: Automate sync to DynamoDB with Lambda

Automate DynamoDB inserts/updates with an S3-triggered Lambda.

**Lambda behavior**:

- Fires on `ObjectCreated` events in the S3 bucket under the `movies/` prefix
- If no entry exists for the movie, create one with the computed partition/sort keys
- Two additional fields:
  - `rank_hist` — a map of `{year_weekN: rank}` entries tracking the movie's ranking position across weeks. Weeks are numbered per ISO calendar (e.g. January 12 2025 is week 2, January 13 2025 is week 3, because January 1 2025 was a Wednesday). For the most popular movie on 2025-01-12 we'd add `rank_hist["2025_02"] = 1`.
  - `until_date` — stores the most recent query date in `yyyy-mm-dd` format. Used to compute how long the movie has been in theaters.

- If an entry already exists, update:
  - `val` — use the latest vote_average
  - `until_date` — latest query date
  - `rank_hist` — add a new entry for the current week

**Implementation steps**:

- Draft the Lambda handler that creates/updates DynamoDB entries from the S3-event JSON
- Deploy via the AWS Console with an S3 trigger (bucket + `movies/` prefix, all ObjectCreated events)
- Use `boto3` to read the object's contents from S3 (cell 4.2 has helper code)
- Test by generating new S3 objects (run earlier cells that call the API)

<div class="alert alert-block alert-info">
💡 Using `update_item` with an `UpdateExpression` is preferred over a read-modify-write pattern — it's atomic, cheaper, and avoids race conditions.
</div>

In [67]:
import os
import requests
import json
import boto3
from decimal import Decimal
from datetime import datetime

API_KEY = os.environ.get("TMDB_API_KEY", "<YOUR_TMDB_API_KEY>")
NOMBRE_TABLA = "MoviesDB"

# 1.- OBTENER DATOS DEL API MOVIES DB
response = requests.get(
    f"https://api.themoviedb.org/3/movie/now_playing?api_key={API_KEY}&language=es-ES&page=1"
)

if response.status_code == 200:
    data = response.json()
    peliculas = data['results']
    for idx, pelicula in enumerate(peliculas):
        pelicula['rank'] = idx + 1
else:
    print(f"Error en la consulta: {response.status_code}")
    peliculas = []

# 2.- CONFIGURAR EL RECURSO Y TABLA DE DYNAMODB
dynamodb_resource = session.resource('dynamodb', region_name='us-east-1')
dynamodb_table = dynamodb_resource.Table(NOMBRE_TABLA)

# 3.- MODIFICAR FORMATO PARA DINAMODB Y GUARDAR/ACTUALIZAR SOLO LA PRIMERA PELÍCULA
if peliculas:
    now = datetime.now()
    today_str = now.strftime("%Y-%m-%d")
    year, week, _ = now.isocalendar()
    week_str = f"w{year}_{week:02d}"

    first_movie = peliculas[0]
    movie_id = str(first_movie["id"])
    release_date = first_movie.get("release_date", "0000-00-00")

    try:
        fecha = datetime.strptime(release_date, "%Y-%m-%d")
        y_m = fecha.strftime("%Y_%m")
    except ValueError:
        y_m = "unknown"

    val = Decimal(str(first_movie.get("vote_average", 0)))
    rank = int(first_movie.get("rank", 0))

    # 4.- LÓGICA DE CREACIÓN O ACTUALIZACIÓN DE ENTRADA
    # Comprobar si ya existe el campo rank_hist
    item = dynamodb_table.get_item(Key={"id": movie_id}).get('Item', {})

    if 'rank_hist' not in item:
        # Si no existe, crea el mapa rank_hist con la semana actual
        rank_hist = {week_str: rank}
        update_expr = (
            "SET y_m = :y_m, val = :val, until_date = :until_date, rank_hist = :rank_hist"
        )
        expr_attr_vals = {
            ":y_m": y_m,
            ":val": val,
            ":until_date": today_str,
            ":rank_hist": rank_hist
        }
        dynamodb_table.update_item(
            Key={"id": movie_id},
            UpdateExpression=update_expr,
            ExpressionAttributeValues=expr_attr_vals
        )
    else:
        # Si existe, actualiza solo el campo de la semana en rank_hist
        update_expr = (
            "SET y_m = :y_m, val = :val, until_date = :until_date, rank_hist.#week = :rank"
        )
        expr_attr_vals = {
            ":y_m": y_m,
            ":val": val,
            ":until_date": today_str,
            ":rank": rank
        }
        expr_attr_names = {
            "#week": week_str
        }
        dynamodb_table.update_item(
            Key={"id": movie_id},
            UpdateExpression=update_expr,
            ExpressionAttributeValues=expr_attr_vals,
            ExpressionAttributeNames=expr_attr_names
        )

    print(f"✅ Película {first_movie['title']} creada/actualizada correctamente en DynamoDB.")

✅ Película Una película de Minecraft creada/actualizada correctamente en DynamoDB.


### Validation

- Verify the Lambda fires when you upload a movie JSON to S3
- Confirm the function completes without errors
- Force-generate new S3 objects via earlier cells
- Capture screenshots:
  - Lambda monitoring dashboard (last 15 minutes) → `img/eje_4_a.png`
  - DynamoDB table contents → `img/eje_4_b.png`
- Paste the final Lambda handler into cell 4.3 below

In [ ]:
# CELDA 4.3
import boto3
import json
from decimal import Decimal
from datetime import datetime


def convert_floats_to_decimal(obj):
    if isinstance(obj, list):
        return [convert_floats_to_decimal(x) for x in obj]
    elif isinstance(obj, dict):
        return {k: convert_floats_to_decimal(v) for k, v in obj.items()}
    elif isinstance(obj, float):
        return Decimal(str(obj))
    else:
        return obj


def lambda_handler(event, context):
    s3_resource = boto3.resource('s3', region_name='us-east-1')
    now = datetime.now()
    year_number = now.isocalendar()[0]
    week_number = now.isocalendar()[1]
    week_str = f"w{year_number}_{week_number:02d}"
    today_str = now.strftime("%Y-%m-%d")
    
    NOMBRE_TABLA = "MoviesDB"
    dynamodb_resource = boto3.resource('dynamodb', region_name='us-east-1')
    dynamodb_table = dynamodb_resource.Table(NOMBRE_TABLA)
    
    for record in event.get('Records', []):
        bucket = record["s3"]["bucket"]["name"]
        key = record["s3"]["object"]["key"]
        obj = s3_resource.Object(bucket, key)
        element = json.load(obj.get()['Body'], parse_float=Decimal)
        
        peliculas = element if isinstance(element, list) else [element]
        
        for pelicula in peliculas:
            movie_id = str(pelicula["id"])
            release_date = pelicula.get("release_date", "0000-00-00")
            try:
                fecha = datetime.strptime(release_date, "%Y-%m-%d")
                y_m = fecha.strftime("%Y_%m")
            except ValueError:
                y_m = "unknown"
            val = Decimal(str(pelicula.get("vote_average", 0)))
            rank = int(pelicula.get("rank", 0))

            item = dynamodb_table.get_item(Key={"id": movie_id}).get('Item', {})

            if not item:
                # Es nuevo: guarda TODO el objeto + especiales
                pelicula['id'] = movie_id
                pelicula['y_m'] = y_m
                pelicula['val'] = val
                pelicula['until_date'] = today_str
                pelicula['rank_hist'] = {week_str: rank}
                item_to_save = convert_floats_to_decimal(pelicula)
                dynamodb_table.put_item(Item=item_to_save)
            else:
                # Ya existe: solo actualiza los campos especiales
                update_expr = (
                    "SET y_m = :y_m, val = :val, until_date = :until_date, rank_hist.#week = :rank"
                )
                expr_attr_vals = {
                    ":y_m": y_m,
                    ":val": val,
                    ":until_date": today_str,
                    ":rank": rank
                }
                expr_attr_names = {
                    "#week": week_str
                }
                dynamodb_table.update_item(
                    Key={"id": movie_id},
                    UpdateExpression=update_expr,
                    ExpressionAttributeValues=expr_attr_vals,
                    ExpressionAttributeNames=expr_attr_names
                )
    return {
        'statusCode': 200,
        'body': 'Películas procesadas correctamente'
    }


<img src="img/eje_4_a.png" alt="Validacion 4a">

<img src="img/eje_4_b.png" alt="Validacion 4b">

### Step 5: Schedule the API polling

The next step is automating the API-to-S3 step itself. Deploy a Lambda with similar logic to the notebook cell that hits the API and writes to S3, then wire up a 24-hour EventBridge trigger.

The Lambda will need:

- The `requests` library, which isn't in Lambda's default runtime — package it as a Lambda layer
- A higher timeout and more memory than the defaults (the API call + multiple S3 writes can exceed 3 seconds)

**Steps**:

- Deploy the API-polling function, similar to Step 2
- Create the `requests` Lambda layer and attach it to the function
- Tune timeout and memory if needed
- Add an EventBridge rule to trigger the function once per day

### Building the Lambda layer (Linux/Mac — adapt for Windows)

```bash
virtualenv --python="/usr/bin/python3.12" .venv
source .venv/bin/activate
mkdir python
cd python/
pip install requests "urllib3<2" -t .
cd ..
zip -r python_modules.zip python/
```

Upload `python_modules.zip` as a Lambda layer.

### Validation

- Confirm the function runs without errors
- Temporarily change the trigger to fire every minute and let it run for 5 minutes
- Capture a screenshot of the monitoring dashboard → `img/eje_5.png`
- Revert the trigger to once-daily

<img src="img/eje_5_sol.png" alt="Validacion 5">

### Step 6: List movies by month — API endpoint

Build a Lambda that accepts events shaped like `{'pathParameters': {'month': '1', 'year': '2025'}}` and returns the top 10 highest-rated movies from that month.

**Query details**:

- Use DynamoDB's `query` operation on the Global Secondary Index
- Filter on `y_m` for the year/month combination
- Sort descending on the sort key (`val`) for highest-rated first
- Limit to 10 movies
- Wrap the result as JSON in the `body` field of the response (API Gateway convention)

**Local test**:

```python
lambda_handler({'pathParameters': {'month': '2', 'year': '2025'}}, None)
```

**API Gateway deployment**:

- Define the resource tree `/list/{year}/{month}`
- Attach a `GET` method on `{month}` that invokes the Lambda with Lambda Proxy Integration enabled
- Deploy a stage (e.g. `test` or `dev`)
- Test with Postman against `<API_URL>/<stage>/list/2025/2`

In [78]:
import boto3
import decimal
import json
from boto3.dynamodb.conditions import Key

NOMBRE_TABLA = "MoviesDB"
NOMBRE_GSI = "y_m-val-index"  # Cambia si tu GSI tiene otro nombre

dynamo_resource = session.resource('dynamodb', region_name='us-east-1')
dynamo_table = dynamo_resource.Table(NOMBRE_TABLA)

class DecimalEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, decimal.Decimal):
            return float(o)
        return super(DecimalEncoder, self).default(o)
    
def lambda_handler(event, context):
    year = int(event['pathParameters']['year'])
    month = int(event['pathParameters']['month'])
    y_m = f'{year}_{month:02d}'
    
    # Consulta sobre el GSI (y_m como partition key, val como sort key descendente)
    response = dynamo_table.query(
        IndexName=NOMBRE_GSI,
        KeyConditionExpression=Key('y_m').eq(y_m),
        ScanIndexForward=False,    # Orden descendente (mayor valoración primero)
        Limit=10                   # Solo los 10 primeros
    )
    items = response.get('Items', [])
    
    return {
        'statusCode': 200,
        'body': json.dumps(items, cls=DecimalEncoder)
    }

In [80]:
# VALIDACIÓN 

event = {'pathParameters': {'month': '2', 'year': '2025'}} # Cambiad el mes si fuera necesario
lambda_handler(event, {})

{'statusCode': 200,
 'body': '[{"original_title": "In the Lost Lands", "genre_ids": [28.0, 14.0, 12.0], "val": 6.407, "adult": false, "y_m": "2025_02", "overview": "Basada en el relato de George R. R. Martin. Una reina (Amara Okereke), desesperada por encontrar la felicidad en el amor, env\\u00eda a la poderosa bruja Gray Alys (Milla Jovovich) a las Tierras Perdidas, en busca de un poder m\\u00e1gico que permite a una persona transformarse en un hombre lobo. Con el misterioso cazador Boyce (Dave Bautista), que la apoya en la lucha contra criaturas oscuras y despiadadas, Gray deambula por un mundo inquietante y peligroso. Pero solo ella sabe que, cada deseo que se concede, tiene consecuencias inimaginables.", "vote_average": 6.407, "rank": 17.0, "rank_hist": {"w2025_22": 17.0}, "popularity": 142.6165, "backdrop_path": "/op3qmNhvwEvyT7UFyPbIfQmKriB.jpg", "release_date": "2025-02-27", "original_language": "en", "vote_count": 391.0, "until_date": "2025-05-27", "id": "324544", "poster_path"

### Step 7: Movie details by ID — API endpoint

Build a second Lambda that accepts `{'pathParameters': {'id': '1300607'}}` and returns full details for the given movie ID.

**Query details**:

- Use DynamoDB `get_item` on the main table (not the GSI)
- Query by partition key (`id`)
- Wrap the result as JSON in the `body` field

**Local test**:

```python
lambda_handler({'pathParameters': {'id': '1300607'}}, None)
```

**API Gateway deployment**:

- Add resource tree `/movies/{id}` to the same API
- Attach a `GET` method on `{id}` that invokes the Lambda with Lambda Proxy Integration
- Redeploy the stage
- Test with Postman

In [83]:
# CELDA 7.1

import boto3
import decimal
import json
from boto3.dynamodb.conditions import Key

NOMBRE_TABLA = "MoviesDB"
dynamo_resource = session.resource('dynamodb', region_name='us-east-1')
dynamo_table = dynamo_resource.Table(NOMBRE_TABLA)


class DecimalEncoder(json.JSONEncoder):       # Necesario para manejar los tipos Decimal
    def default(self, o):
        if isinstance(o, decimal.Decimal):
            return str(o)
        return super(DecimalEncoder, self).default(o)
    
    
def lambda_handler(event, context):
    movie_id = event['pathParameters']['id']
    response = dynamo_table.get_item(Key={'id': movie_id})
    item = response.get('Item')
    if item:
        return {
            'statusCode': 200,
            'body': json.dumps(item, cls=DecimalEncoder)
        }
    else:
        return {
            'statusCode': 404,
            'body': json.dumps({'message': f'No se encontro pelicula con id {movie_id}'})
        }

In [85]:
# VALIDACIÓN 

event = {'pathParameters': {'id': '1241436'}}
lambda_handler(event, {})

{'statusCode': 200,
 'body': '{"original_title": "Warfare", "genre_ids": ["10752", "28"], "val": "7.241", "adult": false, "y_m": "2025_04", "overview": "Basada en las experiencias reales del ex marine Ray Mendoza (codirector y coguionista de la pel\\u00edcula) durante la guerra de Irak. Introduce al espectador en la experiencia de un pelot\\u00f3n de Navy SEALs estadounidenses. Concretamente en una misi\\u00f3n de vigilancia que se tuerce en territorio insurgente. Una historia visceral y a pie de campo sobre la guerra moderna y la hermandad, contada como nunca antes: en tiempo real y basada en los recuerdos de quienes la vivieron.", "vote_average": "7.241", "rank": "8", "rank_hist": {"w2025_22": "8"}, "popularity": "239.3911", "backdrop_path": "/cJvUJEEQ86LSjl4gFLkYpdCJC96.jpg", "release_date": "2025-04-09", "original_language": "en", "vote_count": "433", "until_date": "2025-05-27", "id": "1241436", "poster_path": "/fkVpNJugieKeTu7Se8uQRqRag2M.jpg", "video": false, "title": "Warfare. T

### Step 8: Interactive validation with IPyWidgets

To validate both endpoints end-to-end, the notebook provides a set of IPyWidgets that make API calls from inside Jupyter. Simply set `BASE_URL` to your deployed API endpoint (including stage) in the cell below.

**Steps**:

- Verify with the pre-configured solution URL first (if provided by the instructor)
- Then update `BASE_URL` to your own deployed API and re-validate

In [1]:
import os
# Original instructor-provided validation URL removed.
BASE_URL = os.environ.get('API_GATEWAY_URL', '<YOUR_API_GATEWAY_URL>')  # e.g. https://abc123.execute-api.us-east-1.amazonaws.com/dev

<div class="alert alert-block alert-warning">
⚠️ The cell below doesn't need modification — it uses the `BASE_URL` defined in the previous cell.
</div>

In [2]:
%matplotlib inline

import requests
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from matplotlib.ticker import MaxNLocator    
    
def display_movie_info(year, month):    
    url = BASE_URL + f'/list/{year}/{month}'
    res = requests.get(url)
    data = res.json()
    
    dropdown_movies = widgets.Dropdown(
        options=[(x['title'], x['id']) for x in data],    
        description='Pelicula'
    )
    display(dropdown_movies)
    movie_data = widgets.interactive_output(display_movie, {'_id': dropdown_movies})
    display(movie_data)
    
def display_movie(_id):
    url = BASE_URL + f'/movies/{_id}'
    res = requests.get(url)
    data = res.json()
    if data:
        output1 = widgets.Output()
        with output1:
            poster_path = 'http://image.tmdb.org/t/p/w185'+data['poster_path']
            html = HTML(f'''
                <p>
                    <b>Estreno:</b> {data["release_date"]}</br>
                    <b>Hasta:</b> {data["until_date"]}</br>
                    <b>Votos:</b> {data["vote_count"]}</br>                    
                    <b>Media:</b> {data["vote_average"]}
                </p>
                <img src="{poster_path}" style=max-width:185px;"/>
            ''')
            display(html)

        output2 = widgets.Output()
        with output2:
            ranks = [int(value) for value in data['rank_hist'].values()]
            weeks = list(data['rank_hist'].keys())

            # Configurar grafica
            fig = plt.figure(figsize=(8, 5), dpi=100)
            ax = fig.gca()
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
            plt.xticks(range(len(weeks)), weeks)
            plt.ylim(0, max(ranks) + 1)

            plt.title("Ranking semanal")
            plt.xlabel("Semana")
            plt.ylabel("Posición")

            # Dibujar grafica
            plt.plot(ranks, marker='o', linewidth=2) 
            plt.show()

        two_columns = widgets.HBox([output1, output2])
        display(two_columns)


dropdown_year = widgets.Dropdown(
    options=[2025],    
    description='Año'
)

dropdown_month = widgets.Dropdown(
    options=range(1, 13),    
    description='Mes'
)
movie_info = widgets.interactive_output(display_movie_info, {'year': dropdown_year, 'month': dropdown_month})

display(dropdown_year)
display(dropdown_month)
display(movie_info)

Dropdown(description='Año', options=(2025,), value=2025)

Dropdown(description='Mes', options=(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12), value=1)

Output()

### Step 9 (Optional): CloudWatch Monitoring

CloudWatch can expose a dashboard showing metrics across the app's services (Lambda invocations, DynamoDB read/write capacity, etc.). This is an optional deep-dive for anyone wanting to explore monitoring in more depth.

Build a dashboard showing whichever metrics you consider most useful for operating this application. Build the dashboard entirely through the AWS Console.

### Validation

Screenshot the dashboard and save to `img/eje_9.png`.

<img src="img/eje_9.png" alt="Validacion 9">

<div style="text-align: right">
<a href="#inicio"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#00586D"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#00586D"></i> </font></div>